# # 10. Master Pipeline — All Experiments in One
# 세션 재시작 시 Drive에서 scores 로드 → scoring 스킵 가능
# CANYA는 코랩 TF 호환 불가로 제외 (로컬에서 실행)



In [ ]:
# CELL 1: Setup
import os, sys
os.chdir('/content')
if not os.path.exists('brain_idp_flow'):
    !git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
else:
    !cd brain_idp_flow && git pull origin master
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy scikit-learn openpyxl statsmodels -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch, numpy as np, yaml, glob, os, pickle, json, subprocess
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2: Drive 마운트 + 모델 로드
from google.colab import drive
drive.mount('/content/drive')

from brain_idp_flow.sample import load_model, sample_ensemble_with_trajectory
from brain_idp_flow.features.seq_embed import ESM2Embedder
from brain_idp_flow.geometry.metrics import radius_of_gyration
from brain_idp_flow.data.dataset import ProteinEnsembleDataset
from brain_idp_flow.analysis.trajectory_analysis import extract_trajectory_features
from brain_idp_flow.targets import load_targets

CKPT_35M = '/content/drive/MyDrive/brain_idp_flow_ckpts/ckpt_best.pt'
CKPT_650M = '/content/drive/MyDrive/brain_idp_flow_ckpts_650m/ckpt_best.pt'

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)
with open('configs/flow_650m.yaml') as f:
    config_650m = yaml.safe_load(f)

model = load_model(config, CKPT_35M, device)
embedder = ESM2Embedder(device=device)
print('35M model loaded')

model_650m = load_model(config_650m, CKPT_650M, device)
embedder_650m = ESM2Embedder(model_name='esm2_t33_650M_UR50D', device=device)
print('650M model loaded')

ABETA_SEQ = 'DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA'
IAPP_WT = "KCNTATCATQRLANFLVHSSNNFGAILSSTNVGSNTY"
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

In [ ]:
# CELL 3: Load or Score — Drive 캐시 확인
CACHE_PATH = '/content/drive/MyDrive/brain_idp_flow_scored/all_scores.pkl'
NEED_SCORING = True

if os.path.exists(CACHE_PATH):
    print(f"Found cached scores at {CACHE_PATH}")
    with open(CACHE_PATH, 'rb') as f:
        cached = pickle.load(f)
    for k, v in cached.items():
        globals()[k] = v
        print(f"  Loaded {k}: {len(v)} items")
    NEED_SCORING = False
    print("\nScoring SKIPPED — loaded from Drive cache.")
else:
    print("No cache found. Will run full scoring.")

In [ ]:
# CELL 4: Abeta42 DMS Scoring (캐시 없을 때만)
if NEED_SCORING:
    from brain_idp_flow.data.dms_loader import load_seuma_dms
    os.makedirs('data/dms', exist_ok=True)
    dms_data = load_seuma_dms(cache_dir='data/dms')
    print(f'Abeta42 DMS: {len(dms_data)} mutations')

    # 35M scoring
    wt_emb = embedder.embed_single(ABETA_SEQ)
    wt_result = sample_ensemble_with_trajectory(
        model, wt_emb, mut_pos=0, mut_aa=0,
        n_samples=64, n_trajectory_samples=16, device=device,
    )
    wt_rg = radius_of_gyration(torch.from_numpy(wt_result['ensemble'])).mean().item()
    print(f'WT Rg (35M): {wt_rg:.2f}')

    scored_dms = []
    for i, mut in enumerate(dms_data):
        mut_seq = list(ABETA_SEQ)
        mut_seq[mut['pos'] - 1] = mut['mt']
        mut_seq = ''.join(mut_seq)
        mut_emb = embedder.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
        result = sample_ensemble_with_trajectory(
            model, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
            n_samples=32, n_trajectory_samples=4, n_steps=50, device=device,
        )
        mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).mean().item()
        traj_feats = extract_trajectory_features(result['trajectory'], mutation_pos=mut['pos'] - 1)
        scored_dms.append({
            **mut, 'delta_rg': mut_rg - wt_rg,
            **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
        })
        if (i + 1) % 100 == 0:
            print(f'  35M: {i+1}/{len(dms_data)}')
    print(f'35M Done: {len(scored_dms)}')

    # ESM-2 LLR
    from brain_idp_flow.analysis.esm2_llr import ESM2MutationScorer
    scorer = ESM2MutationScorer(device=device)
    for i, mut in enumerate(scored_dms):
        llr_result = scorer.score_mutation(ABETA_SEQ, mut['pos'], mut['wt'], mut['mt'], fast=True)
        mut['llr_site'] = llr_result['llr_site']
        if (i + 1) % 200 == 0:
            print(f'  LLR {i+1}/{len(scored_dms)}')
    print('LLR done')

    # 650M scoring
    wt_emb_650 = embedder_650m.embed_single(ABETA_SEQ)
    wt_result_650 = sample_ensemble_with_trajectory(
        model_650m, wt_emb_650, mut_pos=0, mut_aa=0,
        n_samples=64, n_trajectory_samples=4, device=device,
    )
    wt_rg_650 = radius_of_gyration(torch.from_numpy(wt_result_650['ensemble'])).mean().item()

    scored_650m = []
    for i, mut in enumerate(dms_data):
        mut_seq = list(ABETA_SEQ)
        mut_seq[mut['pos'] - 1] = mut['mt']
        mut_seq = ''.join(mut_seq)
        mut_emb = embedder_650m.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
        result = sample_ensemble_with_trajectory(
            model_650m, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
            n_samples=32, n_trajectory_samples=4, n_steps=50, device=device,
        )
        mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).mean().item()
        scored_650m.append({**mut, 'delta_rg_650m': mut_rg - wt_rg_650})
        if (i + 1) % 100 == 0:
            print(f'  650M: {i+1}/{len(dms_data)}')
    print(f'650M Done: {len(scored_650m)}')

In [ ]:
# CELL 5: IAPP DMS Scoring (캐시 없을 때만)
if NEED_SCORING:
    # IAPP 데이터 로드
    for fname, url in {
        "MAVE_IAPP_fitness_replicates.RData":
            "https://raw.githubusercontent.com/BEBlab/MAVE-IAPP/main/Required%20files/MAVE_IAPP_fitness_replicates.RData",
        "INDEL_datasets_AB.Rdata":
            "https://raw.githubusercontent.com/BEBlab/MAVE-IAPP/main/Required%20files/INDEL_datasets_AB.Rdata",
    }.items():
        if not os.path.exists(fname):
            subprocess.run(["wget", "-q", url, "-O", fname], check=True)

    r_fix = 'load("MAVE_IAPP_fitness_replicates.RData")\nload("INDEL_datasets_AB.Rdata")\nwrite.csv(singles.df, "iapp_dms_fitness.csv", row.names=FALSE)\ncat("Exported:", nrow(singles.df), "rows\\n")\n'
    with open("_fix.R", "w") as f:
        f.write(r_fix)
    r = subprocess.run(["Rscript", "_fix.R"], capture_output=True, text=True)
    print(r.stdout); os.remove("_fix.R")

    df = pd.read_csv("iapp_dms_fitness.csv", low_memory=False)
    iapp_dms = []
    for _, row in df.iterrows():
        wt = str(row['WT_AA']).strip()
        mt = str(row['Mut']).strip()
        pos = int(row['Pos'])
        if wt not in VALID_AA or mt not in VALID_AA or wt == mt:
            continue
        if pos < 1 or pos > len(IAPP_WT):
            continue
        ns = row['nscore_c']
        if pd.isna(ns):
            continue
        sigma = float(row['sigma']) if not pd.isna(row['sigma']) else 0.0
        iapp_dms.append({
            'pos': pos, 'wt': wt, 'mt': mt,
            'mutation_id': f"{wt}{pos}{mt}",
            'nucleation_score': float(ns), 'sigma': sigma,
            'agg_rate': float(np.exp(np.clip(float(ns), -5, 5))),
            'is_fad': bool(row.get('fAD', False)),
            'target': 'iapp', 'source': 'Badia2026',
        })
    print(f'IAPP DMS loaded: {len(iapp_dms)} substitutions')

    # IAPP 35M scoring
    wt_emb_iapp = embedder.embed_single(IAPP_WT)
    wt_res_iapp = sample_ensemble_with_trajectory(
        model, wt_emb_iapp, mut_pos=0, mut_aa=0,
        n_samples=64, n_trajectory_samples=0, device=device,
    )
    wt_rg_mean_iapp = radius_of_gyration(torch.from_numpy(wt_res_iapp['ensemble'])).numpy().mean()

    scored_iapp_35m = []
    for i, mut in enumerate(iapp_dms):
        mut_seq = list(IAPP_WT)
        mut_seq[mut['pos'] - 1] = mut['mt']
        mut_seq = ''.join(mut_seq)
        mut_emb = embedder.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
        result = sample_ensemble_with_trajectory(
            model, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
            n_samples=32, n_trajectory_samples=0, device=device,
        )
        mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).numpy()
        scored_iapp_35m.append({**mut, 'delta_rg': mut_rg.mean() - wt_rg_mean_iapp})
        if (i + 1) % 100 == 0:
            print(f"  IAPP 35M: {i+1}/{len(iapp_dms)}")
    print(f"IAPP 35M Done: {len(scored_iapp_35m)}")

    # IAPP 650M scoring
    wt_emb_iapp_650 = embedder_650m.embed_single(IAPP_WT)
    wt_res_iapp_650 = sample_ensemble_with_trajectory(
        model_650m, wt_emb_iapp_650, mut_pos=0, mut_aa=0,
        n_samples=64, n_trajectory_samples=0, device=device,
    )
    wt_rg_mean_iapp_650 = radius_of_gyration(torch.from_numpy(wt_res_iapp_650['ensemble'])).numpy().mean()

    scored_iapp_650m = []
    for i, mut in enumerate(iapp_dms):
        mut_seq = list(IAPP_WT)
        mut_seq[mut['pos'] - 1] = mut['mt']
        mut_seq = ''.join(mut_seq)
        mut_emb = embedder_650m.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
        result = sample_ensemble_with_trajectory(
            model_650m, mut_emb, mut_pos=mut['pos'], mut_aa=aa_idx,
            n_samples=32, n_trajectory_samples=0, device=device,
        )
        mut_rg = radius_of_gyration(torch.from_numpy(result['ensemble'])).numpy()
        scored_iapp_650m.append({**mut, 'delta_rg_650m': mut_rg.mean() - wt_rg_mean_iapp_650})
        if (i + 1) % 100 == 0:
            print(f"  IAPP 650M: {i+1}/{len(iapp_dms)}")
    print(f"IAPP 650M Done: {len(scored_iapp_650m)}")

    # Cross-protein transfer
    targets = load_targets('configs/targets.yaml')
    disease_data = []
    for tid, target in targets.items():
        if tid == 'abeta42':
            continue
        wt_emb_t = embedder.embed_single(target.sequence)
        wt_res_t = sample_ensemble_with_trajectory(
            model, wt_emb_t, mut_pos=0, mut_aa=0,
            n_samples=64, n_trajectory_samples=16, device=device,
        )
        wt_rg_t = radius_of_gyration(torch.from_numpy(wt_res_t['ensemble'])).mean().item()
        for mut in target.mutations:
            mut_seq = target.mutant_sequence(mut)
            mut_emb = embedder.embed_single(mut_seq)
            aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut.mt, 0)
            res = sample_ensemble_with_trajectory(
                model, mut_emb, mut_pos=mut.pos, mut_aa=aa_idx,
                n_samples=32, n_trajectory_samples=4, device=device,
            )
            mut_rg = radius_of_gyration(torch.from_numpy(res['ensemble'])).mean().item()
            traj_feats = extract_trajectory_features(res['trajectory'], mutation_pos=mut.pos - 1)
            llr = scorer.score_mutation(target.sequence, mut.pos, mut.wt, mut.mt, fast=True)
            disease_data.append({
                'target': tid, 'mutation_id': mut.id,
                'pos': mut.pos, 'wt': mut.wt, 'mt': mut.mt,
                'agg_rate': mut.agg_rate_relative,
                'delta_rg': mut_rg - wt_rg_t, 'llr_site': llr['llr_site'],
                **{k: v for k, v in traj_feats.items() if not k.startswith('_')},
            })

In [ ]:
# CELL 6: Save scores to Drive (scoring 완료 후 한 번만)
if NEED_SCORING:
    SAVE_DIR = '/content/drive/MyDrive/brain_idp_flow_scored'
    os.makedirs(SAVE_DIR, exist_ok=True)
    data_to_save = {}
    for var_name in ['scored_dms', 'scored_650m', 'scored_iapp_35m', 'scored_iapp_650m',
                     'dms_data', 'iapp_dms', 'disease_data']:
        if var_name in globals() and globals()[var_name]:
            data_to_save[var_name] = globals()[var_name]
    with open(f'{SAVE_DIR}/all_scores.pkl', 'wb') as f:
        pickle.dump(data_to_save, f)
    print(f"Saved {len(data_to_save)} variables to Drive")
    for k, v in data_to_save.items():
        print(f"  {k}: {len(v)} items")

In [ ]:
# CELL 7: Correlation Table + 35M vs 650M Plot
os.makedirs('results/dms', exist_ok=True)
ns_ab = np.array([d['nucleation_score'] for d in scored_dms])
drg_ab = np.array([d['delta_rg'] for d in scored_dms])
drg_ab_650 = np.array([d['delta_rg_650m'] for d in scored_650m])
llr_ab = np.array([d.get('llr_site', 0) for d in scored_dms])
is_fad = np.array([d.get('is_fad', False) for d in scored_dms])

print('=' * 60)
print(f'ABETA42 DMS CORRELATION (n={len(scored_dms)})')
print('=' * 60)
for feat_key, feat_name in [
    ('llr_site', 'ESM-2 LLR'), ('delta_rg', 'Delta Rg (35M)'),
    ('late_velocity_site', 'Late-Stage Velocity'), ('velocity_variance_late', 'Velocity Variance'),
    ('switching_rate_site', 'Contact Switching'), ('switching_rate_long_range', 'Long-Range Switching'),
    ('convergence_time_site', 'Convergence Time'),
]:
    vals = np.array([d.get(feat_key, 0) for d in scored_dms])
    if vals.std() < 1e-10:
        continue
    rho, pval = spearmanr(vals, ns_ab)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'  {feat_name:<30} rho={rho:>7.3f}  p={pval:.2e} {sig}')
rho_650, p_650 = spearmanr(drg_ab_650, ns_ab)
print(f'  {"Delta Rg (650M)":<30} rho={rho_650:>7.3f}  p={p_650:.2e} ***')

# Plot
rho_35m, p_35m = spearmanr(drg_ab, ns_ab)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, drg, rho, p, title in [
    (axes[0], drg_ab, rho_35m, p_35m, 'ESM-2 35M Flow Model'),
    (axes[1], drg_ab_650, rho_650, p_650, 'ESM-2 650M Flow Model'),
]:
    ax.scatter(drg[~is_fad], ns_ab[~is_fad], s=10, alpha=0.4, c='steelblue')
    if is_fad.any():
        ax.scatter(drg[is_fad], ns_ab[is_fad], s=80, facecolors='none',
                   edgecolors='red', linewidth=2, label='fAD')
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title}\nrho={rho:.3f}, p={p:.2e}'); ax.legend(fontsize=9)
fig.suptitle('35M vs 650M: delta Rg vs Nucleation Score', fontsize=14)
fig.tight_layout(); fig.savefig('results/dms/35m_vs_650m.png', dpi=150); plt.show()

In [ ]:
# CELL 8: ML Composite + Cross-Protein Transfer
from brain_idp_flow.analysis.ml_predictor import run_lean_composite, run_full_ml_pipeline, run_cross_protein_transfer

os.makedirs('results/ml', exist_ok=True)
lean_results = run_lean_composite(scored_dms, output_dir='results/ml')
print()
dms_ml_results = run_full_ml_pipeline(scored_dms, output_dir='results/ml')

if 'disease_data' in globals() and disease_data:
    all_combined = scored_dms + disease_data
    print(f'\nCombined: {len(all_combined)} mutations')
    transfer_results = run_cross_protein_transfer(all_combined)

In [ ]:
# CELL 9: Multi-Amyloid Comparison (Abeta42 vs IAPP)
ns_iapp = np.array([d['nucleation_score'] for d in scored_iapp_35m])
drg_iapp = np.array([d['delta_rg'] for d in scored_iapp_35m])
drg_iapp_650 = np.array([d['delta_rg_650m'] for d in scored_iapp_650m])

print("=" * 60)
print("MULTI-AMYLOID delta_Rg CORRELATION")
print("=" * 60)
print(f"{'Protein':<12} {'Model':<8} {'n':>6} {'rho':>8} {'p-value':>12}")
print("-" * 60)
for name, drg, ns_arr in [
    ('Abeta42', drg_ab, ns_ab), ('Abeta42', drg_ab_650, ns_ab),
    ('IAPP', drg_iapp, ns_iapp), ('IAPP', drg_iapp_650, ns_iapp),
]:
    mdl = '650M' if np.array_equal(drg, drg_ab_650) or np.array_equal(drg, drg_iapp_650) else '35M'
    prot = 'Abeta42' if np.array_equal(ns_arr, ns_ab) else 'IAPP'
    rho, p = spearmanr(drg, ns_arr)
    print(f"  {prot:<12} {mdl:<8} {len(ns_arr):>6} {rho:>8.3f} {p:>12.2e}")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for col, (drg, ns_arr, title, c) in enumerate([
    (drg_ab, ns_ab, 'Abeta42 (35M)', 'steelblue'),
    (drg_iapp, ns_iapp, 'IAPP (35M)', 'steelblue'),
]):
    ax = axes[0, col]; rho, p = spearmanr(drg, ns_arr)
    ax.scatter(drg, ns_arr, s=10, alpha=0.4, c=c)
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title} (n={len(ns_arr)})\nrho={rho:.3f}, p={p:.2e}')
for col, (drg, ns_arr, title, c) in enumerate([
    (drg_ab_650, ns_ab, 'Abeta42 (650M)', 'coral'),
    (drg_iapp_650, ns_iapp, 'IAPP (650M)', 'coral'),
]):
    ax = axes[1, col]; rho, p = spearmanr(drg, ns_arr)
    ax.scatter(drg, ns_arr, s=10, alpha=0.4, c=c)
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title} (n={len(ns_arr)})\nrho={rho:.3f}, p={p:.2e}')
fig.suptitle('Multi-Amyloid Validation: Abeta42 vs IAPP', fontsize=16)
fig.tight_layout(); fig.savefig('results/dms/abeta42_vs_iapp.png', dpi=150); plt.show()

# Direction Asymmetry
print("\nDIRECTION ASYMMETRY")
for name, drg, ns_arr in [('Abeta42', drg_ab, ns_ab), ('IAPP', drg_iapp, ns_iapp)]:
    for label, mask in [('accelerating', ns_arr > 0), ('slowing', ns_arr < 0)]:
        if mask.sum() > 10:
            rho, p = spearmanr(drg[mask], ns_arr[mask])
            print(f"  {name} {label} (n={mask.sum()}): rho={rho:.3f}, p={p:.2e}")

In [ ]:
# CELL 10: Bootstrap CI + Multiple Comparison Correction
from statsmodels.stats.multitest import multipletests

def bootstrap_spearman(x, y, n_boot=10000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(x)
    rhos = np.array([spearmanr(x[rng.randint(0,n,n)], y[rng.randint(0,n,n)])[0] for _ in range(n_boot)])
    rhos.sort()
    rho_obs, p_obs = spearmanr(x, y)
    return rho_obs, p_obs, rhos[int(0.025*n_boot)], rhos[int(0.975*n_boot)]

all_tests = [
    ('Abeta42 ESM-2 LLR', llr_ab, ns_ab),
    ('Abeta42 dRg 35M', drg_ab, ns_ab),
    ('Abeta42 dRg 650M', drg_ab_650, ns_ab),
    ('Abeta42 late_velocity', np.array([d.get('late_velocity_site',0) for d in scored_dms]), ns_ab),
    ('Abeta42 vel_variance', np.array([d.get('velocity_variance_late',0) for d in scored_dms]), ns_ab),
    ('Abeta42 switching_site', np.array([d.get('switching_rate_site',0) for d in scored_dms]), ns_ab),
    ('Abeta42 switching_long', np.array([d.get('switching_rate_long_range',0) for d in scored_dms]), ns_ab),
    ('Abeta42 convergence', np.array([d.get('convergence_time_site',0) for d in scored_dms]), ns_ab),
    ('IAPP dRg 35M', drg_iapp, ns_iapp),
    ('IAPP dRg 650M', drg_iapp_650, ns_iapp),
]

print("\n" + "=" * 70)
print("Bootstrap 95% CI + BH Correction")
print("=" * 70)

boot_results = []
for name, x, y in all_tests:
    if np.std(x) < 1e-10:
        continue
    rho, p, ci_lo, ci_hi = bootstrap_spearman(x, y)
    boot_results.append({'name': name, 'rho': rho, 'p': p, 'ci_lo': ci_lo, 'ci_hi': ci_hi})

p_vals = [r['p'] for r in boot_results]
_, p_bh, _, _ = multipletests(p_vals, method='fdr_bh')
_, p_bonf, _, _ = multipletests(p_vals, method='bonferroni')

print(f"{'Test':<30} {'rho':>7} {'CI':>18} {'BH_p':>12} {'Sig':>5}")
print("-" * 75)
for i, r in enumerate(boot_results):
    sig = '***' if p_bh[i] < 0.001 else '**' if p_bh[i] < 0.01 else '*' if p_bh[i] < 0.05 else ''
    print(f"  {r['name']:<30} {r['rho']:>7.3f} [{r['ci_lo']:>.3f}, {r['ci_hi']:>.3f}] {p_bh[i]:>12.2e} {sig:>5}")

# Forest plot
os.makedirs('results/stats', exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 8))
br_sorted = sorted(boot_results, key=lambda x: x['rho'])
for i, r in enumerate(br_sorted):
    color = '#e74c3c' if 'IAPP' in r['name'] else '#3498db'
    ax.plot([r['ci_lo'], r['ci_hi']], [i, i], color=color, linewidth=2)
    ax.plot(r['rho'], i, 'o', color=color, markersize=8)
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
ax.set_yticks(range(len(br_sorted)))
ax.set_yticklabels([r['name'] for r in br_sorted])
ax.set_xlabel('Spearman rho (95% Bootstrap CI)')
ax.set_title('Feature Correlations with Nucleation Score')
fig.tight_layout(); fig.savefig('results/stats/forest_plot_ci.png', dpi=150); plt.show()

In [ ]:
# CELL 11: Per-Residue Rg Contribution (35M vs 650M)
def per_residue_rg_contribution(coords):
    com = coords.mean(axis=-2, keepdims=True)
    dists_sq = ((coords - com) ** 2).sum(axis=-1)
    rg_sq = dists_sq.mean(axis=-1, keepdims=True)
    return (dists_sq / (rg_sq + 1e-8)).mean(axis=0)

wt_emb = embedder.embed_single(ABETA_SEQ)
fad_muts = [d for d in scored_dms if d.get('is_fad', False)][:3]

fig, axes = plt.subplots(len(fad_muts), 2, figsize=(16, 4*len(fad_muts)))
if len(fad_muts) == 1: axes = axes.reshape(1, -1)

for row, mut in enumerate(fad_muts):
    mut_seq = list(ABETA_SEQ); mut_seq[mut['pos']-1] = mut['mt']; mut_seq = ''.join(mut_seq)
    aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut['mt'], 0)
    res_35 = sample_ensemble_with_trajectory(model, embedder.embed_single(mut_seq),
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    wt_35 = sample_ensemble_with_trajectory(model, wt_emb,
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    L = min(res_35['ensemble'].shape[1], wt_35['ensemble'].shape[1])
    dc35 = per_residue_rg_contribution(res_35['ensemble'][:,:L,:]) - per_residue_rg_contribution(wt_35['ensemble'][:,:L,:])
    res_650 = sample_ensemble_with_trajectory(model_650m, embedder_650m.embed_single(mut_seq),
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    wt_650 = sample_ensemble_with_trajectory(model_650m, embedder_650m.embed_single(ABETA_SEQ),
        mut_pos=mut['pos'], mut_aa=aa_idx, n_samples=32, n_trajectory_samples=0, device=device)
    L2 = min(res_650['ensemble'].shape[1], wt_650['ensemble'].shape[1])
    dc650 = per_residue_rg_contribution(res_650['ensemble'][:,:L2,:]) - per_residue_rg_contribution(wt_650['ensemble'][:,:L2,:])
    for col, (dc, label) in enumerate([(dc35,'35M'),(dc650,'650M')]):
        ax = axes[row, col]
        r = np.arange(1, len(dc)+1)
        colors = ['#e74c3c' if d>0 else '#3498db' for d in dc]
        ax.bar(r, dc, color=colors, width=0.8, alpha=0.7)
        ax.axvline(x=mut['pos'], color='black', linestyle='--', linewidth=2,
                   label=f"mut: {mut['wt']}{mut['pos']}{mut['mt']}")
        ax.axhline(y=0, color='gray', linewidth=0.5)
        ax.set_xlabel('Residue'); ax.set_ylabel('delta Rg Contribution')
        ax.set_title(f"{label} | {mut['wt']}{mut['pos']}{mut['mt']}"); ax.legend(fontsize=9)
fig.suptitle('Per-Residue delta Rg Contribution: 35M vs 650M', fontsize=14)
fig.tight_layout(); fig.savefig('results/dms/per_residue_rg_35m_vs_650m.png', dpi=150); plt.show()

In [ ]:
# CELL 12: ESMFold Baseline
print("\n" + "=" * 60)
print("ESMFold Dropout Ensemble Baseline")
print("=" * 60)

from brain_idp_flow.baseline.esmfold_infer import generate_esmfold_ensemble

wt_coords_ef, wt_plddt_ef = generate_esmfold_ensemble(ABETA_SEQ, n_samples=50, device=device)
wt_rg_ef = radius_of_gyration(torch.from_numpy(wt_coords_ef)).mean().item()
print(f"WT Rg (ESMFold): {wt_rg_ef:.2f}")

scored_esmfold = []
for i, mut in enumerate(dms_data):
    mut_seq = list(ABETA_SEQ); mut_seq[mut['pos']-1] = mut['mt']; mut_seq = ''.join(mut_seq)
    try:
        coords, plddt = generate_esmfold_ensemble(mut_seq, n_samples=20, device=device)
        mut_rg = radius_of_gyration(torch.from_numpy(coords)).mean().item()
        scored_esmfold.append({**mut, 'delta_rg_esmfold': mut_rg - wt_rg_ef, 'mean_plddt': float(plddt.mean())})
    except Exception as e:
        scored_esmfold.append({**mut, 'delta_rg_esmfold': 0.0, 'mean_plddt': 0.0})
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(dms_data)}")

drg_ef = np.array([d['delta_rg_esmfold'] for d in scored_esmfold])
rho_ef, p_ef = spearmanr(drg_ef, ns_ab)
print(f"\nESMFold delta_Rg: rho={rho_ef:.3f}, p={p_ef:.2e}")

# IAPP ESMFold
wt_coords_iapp_ef, _ = generate_esmfold_ensemble(IAPP_WT, n_samples=50, device=device)
wt_rg_iapp_ef = radius_of_gyration(torch.from_numpy(wt_coords_iapp_ef)).mean().item()

scored_iapp_ef = []
for i, mut in enumerate(iapp_dms):
    mut_seq = list(IAPP_WT); mut_seq[mut['pos']-1] = mut['mt']; mut_seq = ''.join(mut_seq)
    try:
        coords, _ = generate_esmfold_ensemble(mut_seq, n_samples=20, device=device)
        mut_rg = radius_of_gyration(torch.from_numpy(coords)).mean().item()
        scored_iapp_ef.append({**mut, 'delta_rg_esmfold': mut_rg - wt_rg_iapp_ef})
    except:
        scored_iapp_ef.append({**mut, 'delta_rg_esmfold': 0.0})
    if (i + 1) % 100 == 0:
        print(f"  IAPP: {i+1}/{len(iapp_dms)}")

drg_iapp_ef = np.array([d['delta_rg_esmfold'] for d in scored_iapp_ef])
rho_iapp_ef, p_iapp_ef = spearmanr(drg_iapp_ef, ns_iapp)
print(f"IAPP ESMFold delta_Rg: rho={rho_iapp_ef:.3f}, p={p_iapp_ef:.2e}")

# Comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (drg, name, color) in zip(axes, [
    (drg_ab, 'Flow 35M', 'steelblue'), (drg_ef, 'ESMFold', 'coral'), (drg_ab_650, 'Flow 650M', 'seagreen'),
]):
    rho, p = spearmanr(drg, ns_ab)
    ax.scatter(drg[~is_fad], ns_ab[~is_fad], s=10, alpha=0.4, c=color)
    if is_fad.any():
        ax.scatter(drg[is_fad], ns_ab[is_fad], s=80, facecolors='none', edgecolors='red', linewidth=2, label='fAD')
    ax.set_xlabel('delta Rg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{name}\nrho={rho:.3f}, p={p:.2e}'); ax.legend(fontsize=9)
fig.suptitle('Ensemble Method Comparison', fontsize=14)
fig.tight_layout()
os.makedirs('results/baseline', exist_ok=True)
fig.savefig('results/baseline/esmfold_vs_flow.png', dpi=150); plt.show()

In [ ]:
# CELL 13: Final Summary Table
print("\n" + "=" * 70)
print("FINAL MULTI-METHOD MULTI-AMYLOID COMPARISON")
print("=" * 70)
print(f"  {'Method':<35} {'Abeta42':>12} {'IAPP':>12}")
print("-" * 60)
rho_iapp_flow, _ = spearmanr(drg_iapp, ns_iapp)
print(f"  {'Flow delta_Rg (35M)':<35} {spearmanr(drg_ab, ns_ab)[0]:>12.3f} {rho_iapp_flow:>12.3f}")
print(f"  {'Flow delta_Rg (650M)':<35} {spearmanr(drg_ab_650, ns_ab)[0]:>12.3f} {'---':>12}")
print(f"  {'ESMFold delta_Rg':<35} {rho_ef:>12.3f} {rho_iapp_ef:>12.3f}")
print(f"  {'ESM-2 LLR':<35} {spearmanr(llr_ab, ns_ab)[0]:>12.3f} {'---':>12}")
print(f"  {'CANYA (local run)':<35} {'0.115':>12} {'---':>12}")
print(f"  {'All-7 features (GB CV)':<35} {dms_ml_results['rf']['cv_mean_rho']:>12.3f} {'---':>12}")
print("=" * 70)

In [ ]:
# CELL 14: Save everything + Drive backup
final = {
    'abeta42_35m': float(spearmanr(drg_ab, ns_ab)[0]),
    'abeta42_650m': float(spearmanr(drg_ab_650, ns_ab)[0]),
    'abeta42_esmfold': float(rho_ef),
    'iapp_35m': float(rho_iapp_flow),
    'iapp_esmfold': float(rho_iapp_ef),
    'canya': 0.115,
    'ml_lasso_cv': dms_ml_results['lasso']['cv_mean_rho'],
    'ml_rf_cv': dms_ml_results['rf']['cv_mean_rho'],
}
with open('results/final_results.json', 'w') as f:
    json.dump(final, f, indent=2, default=float)

# Save ESMFold scores too
esmfold_save = {'scored_esmfold': scored_esmfold, 'scored_iapp_ef': scored_iapp_ef}
with open('/content/drive/MyDrive/brain_idp_flow_scored/esmfold_scores.pkl', 'wb') as f:
    pickle.dump(esmfold_save, f)

try:
    import shutil
    shutil.copytree('results', '/content/drive/MyDrive/brain_idp_flow_results', dirs_exist_ok=True)
    print('Backed up to Drive')
except:
    pass

print('\n=== MASTER PIPELINE COMPLETE ===')